# Fintech AI Enable Labs - Getting Started

This notebook provides a comprehensive introduction to the Fintech AI Enable Labs toolkit.

## Overview

The Fintech AI Enable Labs is a Python package designed to help developers and researchers work with financial data and implement AI/ML solutions for fintech applications.

### Key Features:
- Financial data processing and analysis
- Technical indicator calculations
- Machine learning model training and evaluation
- Data validation and quality checks
- Sample data generation for testing

## Installation and Setup

First, make sure you have installed the required dependencies:

```bash
pip install -r requirements.txt
```

In [ ]:
# Import required libraries
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import fintech_ai modules
from fintech_ai import FinanceDataProcessor, ModelTrainer, setup_logging
from fintech_ai.utils import create_sample_data, validate_data

# Setup plotting
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (12, 8)

# Setup logging
logger = setup_logging(level="INFO")
print("Setup complete!")

## 1. Data Generation and Loading

Let's start by generating some sample financial data to work with:

In [ ]:
# Generate sample financial data
sample_data = create_sample_data(n_samples=365)  # One year of daily data
df = sample_data['dataframe']

print("Sample Data Summary:")
for key, value in sample_data['summary'].items():
    print(f"- {key}: {value}")

print("\nFirst 5 rows:")
df.head()

## 2. Data Visualization

Let's visualize the price data:

In [ ]:
# Create subplots
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Financial Data Overview', fontsize=16)

# Price chart
axes[0, 0].plot(df['Date'], df['Close'], label='Close Price', color='blue')
axes[0, 0].plot(df['Date'], df['High'], label='High', alpha=0.7, color='green')
axes[0, 0].plot(df['Date'], df['Low'], label='Low', alpha=0.7, color='red')
axes[0, 0].set_title('Price Chart')
axes[0, 0].set_ylabel('Price ($)')
axes[0, 0].legend()
axes[0, 0].tick_params(axis='x', rotation=45)

# Volume chart
axes[0, 1].bar(df['Date'], df['Volume'], alpha=0.7, color='orange')
axes[0, 1].set_title('Trading Volume')
axes[0, 1].set_ylabel('Volume')
axes[0, 1].tick_params(axis='x', rotation=45)

# Price distribution
axes[1, 0].hist(df['Close'], bins=30, alpha=0.7, color='purple')
axes[1, 0].set_title('Price Distribution')
axes[1, 0].set_xlabel('Price ($)')
axes[1, 0].set_ylabel('Frequency')

# Daily returns
daily_returns = df['Close'].pct_change().dropna() * 100
axes[1, 1].hist(daily_returns, bins=30, alpha=0.7, color='teal')
axes[1, 1].set_title('Daily Returns Distribution')
axes[1, 1].set_xlabel('Daily Return (%)')
axes[1, 1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

print(f"Daily return statistics:")
print(f"Mean: {daily_returns.mean():.3f}%")
print(f"Std: {daily_returns.std():.3f}%")
print(f"Min: {daily_returns.min():.3f}%")
print(f"Max: {daily_returns.max():.3f}%")

## 3. Data Processing and Technical Indicators

Now let's process the data and calculate technical indicators:

In [ ]:
# Initialize data processor
processor = FinanceDataProcessor()
processor.load_data(df)

# Calculate technical indicators
processed_data = processor.calculate_technical_indicators()

print("Technical indicators calculated:")
print(f"Dataset shape: {processed_data.shape}")
print(f"New columns: {[col for col in processed_data.columns if col not in df.columns]}")

# Display some statistics
print("\nTechnical Indicator Statistics:")
print(processed_data[['SMA_20', 'SMA_50', 'RSI', 'Volatility']].describe())

## 4. Technical Indicators Visualization

In [ ]:
# Plot technical indicators
fig, axes = plt.subplots(3, 1, figsize=(15, 12))
fig.suptitle('Technical Indicators Analysis', fontsize=16)

# Price with moving averages
axes[0].plot(processed_data['Date'], processed_data['Close'], label='Close Price', color='blue')
axes[0].plot(processed_data['Date'], processed_data['SMA_20'], label='SMA 20', color='orange')
axes[0].plot(processed_data['Date'], processed_data['SMA_50'], label='SMA 50', color='red')
axes[0].set_title('Price with Moving Averages')
axes[0].set_ylabel('Price ($)')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=45)

# RSI
axes[1].plot(processed_data['Date'], processed_data['RSI'], color='purple')
axes[1].axhline(y=70, color='r', linestyle='--', alpha=0.7, label='Overbought (70)')
axes[1].axhline(y=30, color='g', linestyle='--', alpha=0.7, label='Oversold (30)')
axes[1].set_title('Relative Strength Index (RSI)')
axes[1].set_ylabel('RSI')
axes[1].set_ylim(0, 100)
axes[1].legend()
axes[1].tick_params(axis='x', rotation=45)

# Volatility
axes[2].plot(processed_data['Date'], processed_data['Volatility'], color='teal')
axes[2].set_title('Price Volatility (20-day rolling standard deviation)')
axes[2].set_ylabel('Volatility')
axes[2].set_xlabel('Date')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 5. Machine Learning Model Training

Let's train a machine learning model to predict prices:

In [ ]:
# Prepare features
feature_columns = ['SMA_20', 'SMA_50', 'RSI', 'Volatility', 'Volume']
target_column = 'Close'

processor.prepare_features(feature_columns, target_column)

# Get clean data
clean_data = processor.data.dropna()
X = clean_data[feature_columns]
y = clean_data[target_column]

print(f"Training data prepared:")
print(f"Features: {feature_columns}")
print(f"Target: {target_column}")
print(f"Clean samples: {len(clean_data)}")
print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")

In [ ]:
# Train the model
trainer = ModelTrainer()
results = trainer.train_model(X, y, model_type="random_forest")

print("Model Training Results:")
for key, value in results.items():
    if isinstance(value, float):
        print(f"- {key}: {value:.4f}")
    else:
        print(f"- {key}: {value}")

## 6. Model Evaluation and Predictions

In [ ]:
# Make predictions on the entire dataset
predictions = trainer.predict(X)

# Calculate prediction errors
prediction_errors = predictions - y.values
mae = np.mean(np.abs(prediction_errors))
rmse = np.sqrt(np.mean(prediction_errors ** 2))

print(f"Prediction Metrics:")
print(f"Mean Absolute Error (MAE): ${mae:.2f}")
print(f"Root Mean Square Error (RMSE): ${rmse:.2f}")
print(f"Mean Price: ${y.mean():.2f}")
print(f"MAE as % of mean price: {(mae/y.mean())*100:.2f}%")

In [ ]:
# Visualize predictions vs actual values
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Model Evaluation', fontsize=16)

# Predictions vs Actual (scatter plot)
axes[0, 0].scatter(y.values, predictions, alpha=0.6)
axes[0, 0].plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2)
axes[0, 0].set_xlabel('Actual Price ($)')
axes[0, 0].set_ylabel('Predicted Price ($)')
axes[0, 0].set_title('Predictions vs Actual Values')

# Time series comparison (last 50 points)
last_n = 50
recent_dates = clean_data['Date'].tail(last_n)
recent_actual = y.tail(last_n)
recent_pred = predictions[-last_n:]

axes[0, 1].plot(recent_dates, recent_actual, label='Actual', color='blue')
axes[0, 1].plot(recent_dates, recent_pred, label='Predicted', color='red', alpha=0.7)
axes[0, 1].set_title(f'Recent Predictions (Last {last_n} days)')
axes[0, 1].set_ylabel('Price ($)')
axes[0, 1].legend()
axes[0, 1].tick_params(axis='x', rotation=45)

# Prediction errors distribution
axes[1, 0].hist(prediction_errors, bins=30, alpha=0.7, color='orange')
axes[1, 0].axvline(x=0, color='red', linestyle='--')
axes[1, 0].set_title('Prediction Errors Distribution')
axes[1, 0].set_xlabel('Prediction Error ($)')
axes[1, 0].set_ylabel('Frequency')

# Feature importance
try:
    importance = trainer.get_feature_importance()
    bars = axes[1, 1].bar(range(len(feature_columns)), importance, color='green', alpha=0.7)
    axes[1, 1].set_xticks(range(len(feature_columns)))
    axes[1, 1].set_xticklabels(feature_columns, rotation=45)
    axes[1, 1].set_title('Feature Importance')
    axes[1, 1].set_ylabel('Importance Score')
    
    # Add value labels on bars
    for bar, imp in zip(bars, importance):
        axes[1, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                       f'{imp:.3f}', ha='center', va='bottom')
except Exception as e:
    axes[1, 1].text(0.5, 0.5, f'Feature importance\nnot available:\n{str(e)}',
                   ha='center', va='center', transform=axes[1, 1].transAxes)
    axes[1, 1].set_title('Feature Importance (N/A)')

plt.tight_layout()
plt.show()

## 7. Data Validation

Let's validate our data quality:

In [ ]:
# Validate the processed data
validation_results = validate_data(processed_data)

print("Data Validation Results:")
print(f"Valid: {validation_results['valid']}")

if validation_results['errors']:
    print("\nErrors:")
    for error in validation_results['errors']:
        print(f"  - {error}")

if validation_results['warnings']:
    print("\nWarnings:")
    for warning in validation_results['warnings']:
        print(f"  - {warning}")

print("\nData Info:")
for key, value in validation_results['info'].items():
    if key == 'null_values':
        null_count = sum(value.values())
        print(f"  - {key}: {null_count} total null values")
    else:
        print(f"  - {key}: {value}")

## 8. Summary and Next Steps

Congratulations! You've successfully:

1. ✅ Generated sample financial data
2. ✅ Processed and visualized the data
3. ✅ Calculated technical indicators
4. ✅ Trained a machine learning model
5. ✅ Evaluated model performance
6. ✅ Validated data quality

### Next Steps:

- **Real Data Integration**: Replace sample data with real financial data from APIs like Yahoo Finance, Alpha Vantage, or your own data sources
- **Advanced Models**: Experiment with different ML algorithms (XGBoost, Neural Networks, etc.)
- **Feature Engineering**: Create more sophisticated technical indicators and features
- **Backtesting**: Implement trading strategies and backtest them
- **Risk Management**: Add risk metrics and portfolio optimization
- **Real-time Processing**: Set up streaming data processing for live trading applications

### Resources:

- Check the `examples/` directory for more advanced usage examples
- Review the `config/` directory for configuration options
- Explore the source code in `src/fintech_ai/` for deeper understanding
- Run the test suite in `tests/` to ensure everything works correctly